# LLM JSON Response Evaluator Gate

Visible checkpoint for the saved-response parser/evaluator that will score future local Gemma/Qwen JSON classifier outputs.

This notebook reads a synthetic parser fixture and saved evaluator artifacts only. It does not read holdout, load a model, call an LLM, train, or generate answers.

In [1]:
from pathlib import Path
import json

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

EVAL_DIR = PROJECT_ROOT / "data" / "eval" / "llm_json_classifier_dev"
SUMMARY_PATH = EVAL_DIR / "llm_json_classifier_summary_synthetic_parser_fixture.json"
PARSE_STATUS_PATH = EVAL_DIR / "llm_json_classifier_parse_status_synthetic_parser_fixture.csv"

PROJECT_ROOT

WindowsPath('C:/Users/nwagb/Desktop/it_support_ai')

## Fixture Scope

The synthetic fixture proves parser plumbing only. The metrics below are not model benchmark numbers.

In [2]:
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))

pd.DataFrame([summary["counts"]])

,eval_key_cases,response_rows,valid_response_cases,invalid_response_cases,missing_response_cases,extra_response_rows,predictions_scored
0,2,3,1,1,0,1,2


In [3]:
parse_status = pd.read_csv(PARSE_STATUS_PATH)
parse_status

,profile,case_id,parse_status,parse_error
0,llm_json_classifier_metadata_saved_response_eval,fixture-c1,valid,NaN
1,llm_json_classifier_metadata_saved_response_eval,fixture-c2,invalid,ValueError: No JSON object found in response text
2,llm_json_classifier_metadata_saved_response_eval,fixture-extra,extra,response case_id is not present in eval keys


## Metric Smoke

Invalid or missing responses are scored as failed predictions. Extra response rows are reported but not scored.

In [4]:
metric_smoke = {
    "profile": summary["profile"],
    "primary_accuracy": summary["primary_metrics"]["accuracy"],
    "multilabel_micro_f1": summary["multilabel_metrics"]["micro_f1"],
    "primary_hit_at_3": summary["ranked_metrics"]["primary_hit_at_3"],
    "safety_signal_micro_f1": summary["safety_signal_metrics"]["micro_f1"],
}

pd.DataFrame([metric_smoke])

,profile,primary_accuracy,multilabel_micro_f1,primary_hit_at_3,safety_signal_micro_f1
0,llm_json_classifier_metadata_saved_response_eval,0.5,0.8,0.5,0.0


In [5]:
assert summary["stage"] == "llm_json_classifier_response_eval"
assert summary["counts"]["valid_response_cases"] == 1
assert summary["counts"]["invalid_response_cases"] == 1
assert summary["counts"]["extra_response_rows"] == 1
assert set(parse_status["parse_status"]) == {"valid", "invalid", "extra"}

"parser/evaluator fixture gate passed"

'parser/evaluator fixture gate passed'

## Next Gate

The next step can be a tiny explicitly approved local LLM inference smoke that writes saved response rows. The evaluator here should score those rows before any broader dev run.